# Prop-Firm Analysis 

One consistent pipeline built around various strategies. Set the knobs in Cell 2, then
**Run All**. Every cell uses the same `bars`, `E`, `g`, and `strat`.


In [8]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


## Firm rules ($50K)


In [8]:
for k, f in FIRMS.items():
    print(f"{k:17s} {f.name:26s} dd={f.dd_type.value:14s} lock={str(f.trail_lock):7s} "
          f"DLL={str(f.daily_loss_limit):5s}({f.dll_behavior.value:4s}) "
          f"cons={str(f.consistency_pct):4s} min_days={f.min_days}")

apex_eod          Apex (EOD Trailing)        dd=eod_trail      lock=50000   DLL=1000 (hard) cons=None min_days=1
apex_intraday     Apex (Intraday Trailing)   dd=intraday_trail lock=50000   DLL=None (none) cons=None min_days=1
lucid             Lucid (LucidPro)           dd=eod_trail      lock=50000   DLL=1200 (hard) cons=None min_days=1
tradeify_select   Tradeify (Select)          dd=eod_trail      lock=50000   DLL=None (none) cons=0.4  min_days=3
tradeify_growth   Tradeify (Growth)          dd=eod_trail      lock=50000   DLL=1250 (soft) cons=None min_days=1
mffu_rapid        MyFundedFutures (Rapid)    dd=intraday_trail lock=None    DLL=None (none) cons=0.5  min_days=2
topstep           Topstep (Combine)          dd=eod_trail      lock=50000   DLL=None (none) cons=0.5  min_days=2


## Verify Project Directory and Testing KNOBS Setups

In [3]:
import os, sys
SEARCH_ROOT = r"C:\\Users\\2115h\\Desktop\\Trading"
PROJECT_DIR = next((r for r, d, f in os.walk(SEARCH_ROOT)
                    if "Firms.py" in f and "__pycache__" not in r), os.getcwd())
os.chdir(PROJECT_DIR); sys.path.insert(0, PROJECT_DIR)
print("Project dir:", PROJECT_DIR)

Project dir: C:\\Users\\2115h\\Desktop\\Trading\Propfirm Convex Analysis\Rules


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
from Firms import FIRMS, get_firm
from Strategy import EmpiricalStrategy
from Strategy import Timewindow
from Backtest import load_ohlc, backtest_entries, ledger_for_rr, measured_winrate
from Analysis import (risk_geometry_3d, oos_check,
                      plot_risk_geometry_3d, plot_top_cells, summary_table_3d)
from Surface3d import surface_by_stop, scatter_4d, write_html
from Engine import simulate

# Each strategy owns how to build it, its data prep, and its exit defaults.
# NOTE: no more k_stop_atr — the stop is a SWEEP AXIS now (STOP_GRID in cell 6).
STRATEGIES = {
    "kernel_slope": dict(
    build   = lambda p: Timewindow(
                  __import__("strategies.kernel_slope", fromlist=["get_strategy"]).get_strategy(**p),
                  start="09:30", end="11:00"),
    rth_only=False, timeframe="3min", max_hold=40,
    params  = dict(h=8.0, r=8.0, x_0=5, slope_len=3, slope_min=0.3, atr_len=14,
                   reset_daily=True, src="close",
                   gate_next_body=False, next_body_mult=1.5, next_body_min_pts=20.0),
    ),
    "kernel_regime": dict(
        build   = lambda p: Timewindow(
                      __import__("strategies.kernel_er", fromlist=["get_strategy"]).get_strategy(**p),
                      start="09:30", end="11:00"),
        rth_only=False, timeframe="1min", max_hold=40,
        params  = dict(h=8.0, r=8.0, x_0=3, slope_len=3, slope_min=0.15, atr_len=14,
                er_len=10, er_min=0.30, confirm_bars=1, src="close",
                allow_long=True, allow_short=True,
                filter_clear_prior=True, filter_displacement=False, disp_wick_frac=0.20,
                min_body_pts=4.0,          # <- lowered from 15 (candles are small-point)
                use_er=False,
                use_slope=False,           # <- NEW: no slope gate/grey; direction from kernel flip
                body_frac_prev=0.9,
                check_next=True, min_gap_bars=5,
                max_scan_bars=1,           # <- NEW: entry stays near the flip, not late
                sess_start="09:30", sess_end="11:00")
    ),
    "dual_kernel": dict(
        build   = lambda p: Timewindow(
                  __import__("strategies.kernel_slope", fromlist=["get_strategy"]).get_strategy(**p),
                  start="09:30", end="11:00"),
        rth_only=False, timeframe="1min", max_hold=40,
        params = dict(
            hF=4.0, x0F=1, rF=8.0,           # fast kernel (reactive)
            hS=12.0, x0S=3, rS=8.0,          # slow kernel (trend context)
            atr_len=14,
            slope_len_f=1, slope_min_f=0.0,  # fast slope confirm (0=off)
            slope_len_s=3, slope_min_s=0.0,  # slow slope confirm (0=off; slow is against the cross at the cross bar)
            use_disp=True, min_body_pts=0.0, # displacement: cross candle body clears prior wick
            cross_filter="none",             # "none" | "force" | "post_sep"
            force_min=0.2, sep_conf=0.2, sep_wait=2,
            min_gap_bars=5,
            sess_start="09:30", sess_end="11:00",
            allow_long=True, allow_short=True, src="close"),
    ),
    "dual_kernel_v2": dict(
        build   = lambda p: Timewindow(
                  __import__("strategies.kernel_slope", fromlist=["get_strategy"]).get_strategy(**p),
                  start="09:30", end="11:00"),
        rth_only=False, timeframe="1min", max_hold=40,
        params = dict( hF=4.0, x0F=1, rF=8.0,              # fast kernel
            hS=12.0, x0S=3, rS=8.0,            # slow kernel
            atr_len=14,
            slope_len_f=1, slope_min_f=0.0,    # fast slope confirm (0=off)
            slope_len_s=3, slope_min_s=0.0,    # slow slope confirm (0=off, keep low)
            min_body_pts=10.0,                 # entry bar body >= this
            use_wick=True,                     # + close beyond prior wick
            scan_at=True, scan_p1=True, scan_p2=True,   # eligible bars: cross / +1 / +2 (first displacing one fires)
            use_ext=True, ext_bars=3, ext_pts=30.0,     # skip if prior extBars ran >= extPts same-dir
            use_er=False, er_len=10, er_min=0.3,        # ER chop gate (toggle)
            use_cq=False, open_min=0.10,                # cross-quality open-speed (toggle)
            min_gap_bars=5,
            sess_start="09:30", sess_end="11:00",
            allow_long=True, allow_short=True, src="close"),
    ),
                    
}

def build(name, firm, overrides=None):
    """Single build path. Returns (E, cfg) — E is the scorable ledger."""
    cfg = STRATEGIES[name]
    params = {**cfg["params"], **(overrides or {})}
    strat = cfg["build"](params)
    bars = load_ohlc(DATA_PATH, rth_only=cfg["rth_only"],
                     timeframe=cfg["timeframe"], tz="America/New_York")
    E = backtest_entries(strat, bars, k_stop_atr=cfg["k_stop_atr"], max_hold=cfg["max_hold"])
    print(f"[{name}] {E.summary()} | rth_only={cfg['rth_only']} tf={cfg['timeframe']} "
          f"k_stop={cfg['k_stop_atr']}")
    return E, cfg


## Strategy Initialization

In [17]:

STRATEGY   = "dual_kernel_v2"          # <-- switch strategy here, one line
FIRM       = get_firm("topstep")          # <-- switch firm here, one line
DATA_PATH  = "data/NQ.ohlcv-1m (1).parquet"
STOP_GRID  = [25, 30, 35, 40, 45, 50]                       # POINTS — the swept stop axis
RR_GRID    = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 2.75, 3.0]
RISK_PCTS  = np.round(np.arange(0.25, 3.51, 0.25), 2)
MAX_HOLD   = 100
MAX_DAYS_TO_PASS = 10
N_PATHS    = 2000
ALL_FIRMS  = ["apex_eod","apex_intraday","lucid","tradeify_select",
              "tradeify_growth","mffu_rapid","topstep"]

OVERRIDES  = {}     # e.g. {"slope_min": 0.4} to override a registry param
# ============================================================
print("Active strategy:", STRATEGY)

Active strategy: dual_kernel_v2


## Verify Parquest Data Loaded Correctly (Skip after first time)

In [54]:
PERIOD_START = "2021-01-01"
PERIOD_END = "2026-08-20"

bars_p = load_ohlc(DATA_PATH, rth_only=False, timeframe=None, tz="America/New_York")
bars_p = bars_p.loc[PERIOD_START:PERIOD_END]
print ("columns:", list(bars_p.columns), "| index tz:", bars_p.index.tz)

week = bars_p.loc["2021-01-06":"2026-08-20"]
print(f"\n{len(week)} bars in week {week.index.min().date()} -> {week.index.max().date()}")
print(week.iloc[::60].to_string())
print("\nprice sanity:", week["close"].min(), "-", week["close"].max())

columns: ['open', 'high', 'low', 'close'] | index tz: America/New_York

1816885 bars in week 2021-06-27 -> 2026-08-17
                               open      high       low     close
datetime                                                         
2021-06-27 20:00:00-04:00  14355.00  14356.00  14352.75  14353.75
2021-06-27 21:00:00-04:00  14344.00  14344.25  14342.00  14343.00
2021-06-27 22:00:00-04:00  14340.50  14341.50  14340.25  14340.50
2021-06-27 23:00:00-04:00  14345.50  14346.25  14344.75  14346.25
2021-06-28 00:00:00-04:00  14351.00  14352.00  14351.00  14351.25
2021-06-28 01:00:00-04:00  14357.25  14359.25  14357.25  14358.50
2021-06-28 02:00:00-04:00  14361.75  14361.75  14358.75  14359.75
2021-06-28 03:00:00-04:00  14357.00  14362.00  14352.00  14361.00
2021-06-28 04:00:00-04:00  14375.75  14376.00  14373.75  14374.00
2021-06-28 05:00:00-04:00  14373.00  14376.00  14372.00  14376.00
2021-06-28 06:00:00-04:00  14373.75  14373.75  14370.25  14370.75
2021-06-28 07:00:00-04:0

In [ ]:
from Walkforward import walk_forward
bars = load_ohlc(DATA_PATH, rth_only=_cfg["rth_only"],
                 timeframe=_cfg["timeframe"], tz="America/New_York")

## Backtest & Edge Diagnostic


### 1) Entries Simulation

In [77]:
FIXED_DK = dict( hF=4.0, x0F=1, rF=8.0,              # fast kernel
            hS=16.0, x0S=3, rS=8.0,            # slow kernel
            atr_len=14,
            slope_len_f=1, slope_min_f=0.0,    # fast slope confirm (0=off)
            slope_len_s=3, slope_min_s=0.0,    # slow slope confirm (0=off, keep low)
            min_body_pts=0,                 # entry bar body >= this
            use_wick=True,                     # + close beyond prior wick
            scan_at=True, scan_p1=True, scan_p2=True,   # eligible bars: cross / +1 / +2 (first displacing one fires)
            use_ext=False, ext_bars=3, ext_pts=15.0,     # skip if prior extBars ran >= extPts same-dir
            use_er=False, er_len=10, er_min=0.3,        # ER chop gate (toggle)
            use_cq=False, open_min=0.10,                # cross-quality open-speed (toggle)
            min_gap_bars=2,
            sess_start="09:30", sess_end="11:15",
            allow_long=True, allow_short=True, src="close")
strat = __import__("strategies.dual_kernel_v2", fromlist=["get_strategy"]).get_strategy(**FIXED_DK)

In [ ]:
import pandas as pd
bars_dbg = load_ohlc(DATA_PATH, rth_only=True, timeframe="1min", tz="America/New_York")

# the ACTUAL entries the analysis takes (after latch/session/cooldown/Rule1):
idx, direction = strat.entries(bars_dbg)
entries = pd.DataFrame({
    "time":  bars_dbg.index[idx],
    "dir":   ["LONG" if d == 1 else "SHORT" for d in direction],
    "close": bars_dbg["close"].to_numpy()[idx],
}).set_index("time")

# also pull the regime state so you can see WHY, from debug()
dbg = strat.debug(bars_dbg)

sample_days = ["2026-08-06", "2026-08-05", "2026-08-13"]   # use real trading days
for day in sample_days:
    day_entries = entries.loc[entries.index.normalize() == pd.Timestamp(day, tz=entries.index.tz)]
    print(f"\n===== {day} — {len(day_entries)} ENTRY(ies) =====")
    if len(day_entries):
        print(day_entries.round(2).to_string())
    else:
        print("  (no entries)")
        # show regime transitions in-window to see what was skipped and why
        d = dbg.loc[day].between_time("09:30", "11:30")
        trans = d[d.regimeJustOn]
        if len(trans):
            print(f"  {len(trans)} regime transition(s) but no entry — likely Rule 1 / cooldown gated:")
            print(trans[["k", "slope_atr", "regimeOn", "dirUp"]].round(3).to_string())


===== 2026-08-06 — 17 ENTRY(ies) =====
                             dir     close
time                                      
2026-08-06 09:35:00-04:00   LONG  29367.00
2026-08-06 09:42:00-04:00  SHORT  29413.50
2026-08-06 09:54:00-04:00  SHORT  29500.25
2026-08-06 09:59:00-04:00   LONG  29528.25
2026-08-06 10:03:00-04:00  SHORT  29491.00
2026-08-06 10:06:00-04:00   LONG  29547.50
2026-08-06 10:22:00-04:00   LONG  29654.50
2026-08-06 10:24:00-04:00  SHORT  29634.25
2026-08-06 10:30:00-04:00   LONG  29597.75
2026-08-06 10:37:00-04:00  SHORT  29605.25
2026-08-06 10:42:00-04:00   LONG  29611.75
2026-08-06 10:45:00-04:00  SHORT  29570.25
2026-08-06 10:50:00-04:00   LONG  29592.00
2026-08-06 10:54:00-04:00   LONG  29614.75
2026-08-06 11:02:00-04:00  SHORT  29634.75
2026-08-06 11:06:00-04:00   LONG  29649.50
2026-08-06 11:13:00-04:00  SHORT  29657.00

===== 2026-08-05 — 14 ENTRY(ies) =====
                             dir     close
time                                      
2026-08-05 09:39:

In [74]:
# in the EVAL notebook, run ONCE after your backtest:
import pickle
RETURN_ARRAYS = {rho: ledger_for_rr(E, rho)[0] for rho in RRS}
with open("ledgers.pkl", "wb") as f:
    pickle.dump(RETURN_ARRAYS, f)

### 2) Full Scope Analysis

In [78]:
import pandas as pd, numpy as np
from MultiEval import multi_account_replay
from Backtest import measured_winrate, ledger_for_rr

bars_ks = load_ohlc(DATA_PATH, rth_only=False, timeframe="1min", tz="America/New_York")
window  = bars_ks.loc["2026-01-01":]
STOP, RISK, MAXHOLD, RR = 25, 2, 60, 1.5         # <- RR fixed for rotation

# ══ 1) EDGE by RR — single-account, MAE breach NOT applied (per-trade edge; correct) ══
# (MAE only matters near the floor, which is an EVAL-geometry effect, not a per-trade one)
E = backtest_entries(strat, window, stop_points=STOP, max_hold=MAXHOLD, detail=True, one_position=True)
n = len(E.day_ids)
print(f"dual_kernel_v2 2025 | {n} trades over {E.n_days} days ({n/max(E.n_days,1):.2f}/day) | stop={STOP}pt max_hold={MAXHOLD}\n")
print("── EDGE by RR (per-trade quality) ──")
for rr in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]:
    R,_ = ledger_for_rr(E, rr)
    print(f"  RR {rr:<4}: WR {measured_winrate(E,rr):5.1%} | edge {R.mean():+.3f}R | "
          f"{'POSITIVE' if R.mean()>0 else 'negative'}")

# ══ 2) WINNING vs LOSING DAYS distribution (at fixed RR, single-account) ══
R_fixed, _ = ledger_for_rr(E, RR)
tdays = pd.to_datetime(E.entry_time).tz_convert("America/New_York").normalize()
daily = pd.DataFrame({"day": tdays, "R": R_fixed}).groupby("day")["R"].agg(["sum","count"])
daily.columns = ["net_R","n_trades"]
nd = len(daily); wd = int((daily.net_R > 0).sum())
print(f"\n── DAY DISTRIBUTION at RR={RR} (single-account) ──")
print(f"  trading days: {nd} | WIN days: {wd} ({100*wd/nd:.0f}%) | LOSE days: {nd-wd} ({100*(nd-wd)/nd:.0f}%)")
print(f"  ~per 5-day week: {5*wd/nd:.1f} win / {5*(nd-wd)/nd:.1f} lose")
print(f"  avg net_R: win-day {daily[daily.net_R>0].net_R.mean():+.2f} | lose-day {daily[daily.net_R<=0].net_R.mean():+.2f} | overall {daily.net_R.mean():+.3f}")
print(f"  daily net_R: median {daily.net_R.median():+.2f} | p25 {daily.net_R.quantile(.25):+.2f} | p75 {daily.net_R.quantile(.75):+.2f} | worst {daily.net_R.min():+.2f} | best {daily.net_R.max():+.2f}")

# ══ 3) ROTATION with MAE-breach (multi_account_replay applies it internally, near-floor only) ══
# feed the NON-stacked ledger with one_position=False so 5 accounts absorb overlaps
E_rot = backtest_entries(strat, window, stop_points=STOP, max_hold=MAXHOLD, detail=True, one_position=True)
res = multi_account_replay(E_rot, FIRM, risk_pct=RISK, rr=RR,
                           n_accounts=5, max_days=25, restart_on_fail=True)
d = res.n_passed + res.n_blown; e = d + res.n_timeout
p2p = [a.trading_days for a in res.history if a.status == "PASS"]
print(f"\n── ROTATION 5-slot (MAE-breach applied) | stop={STOP}/{RR}R/{RISK}% ──")
print(f"  {res.n_passed}P / {res.n_blown}B / {res.n_timeout}T | "
      f"pass_rate {100*res.n_passed/max(d,1):.0f}% | econ_rate {100*res.n_passed/max(e,1):.0f}%")
if p2p:
    print(f"  days-to-pass: mean {np.mean(p2p):.1f} | min {min(p2p)} | max {max(p2p)}")

attempts = pd.DataFrame([dict(
    slot=a.idx+1, attempt=a.attempt, status=a.status, days=a.trading_days,
    resolved=a.resolved_at.strftime("%Y-%m-%d") if a.resolved_at is not None else "-",
    equity=round(a.final_equity),
) for a in res.history])
print(f"\nAll {len(attempts)} attempts:")
print(attempts.to_string(index=False))

dual_kernel_v2 2025 | 884 trades over 161 days (5.49/day) | stop=25pt max_hold=60

── EDGE by RR (per-trade quality) ──
  RR 0.5 : WR 65.3% | edge -0.021R | negative
  RR 0.75: WR 58.5% | edge +0.023R | POSITIVE
  RR 1.0 : WR 52.6% | edge +0.055R | POSITIVE
  RR 1.25: WR 47.2% | edge +0.065R | POSITIVE
  RR 1.5 : WR 42.3% | edge +0.062R | POSITIVE
  RR 2.0 : WR 33.7% | edge +0.027R | POSITIVE
  RR 2.5 : WR 29.2% | edge +0.054R | POSITIVE
  RR 3.0 : WR 25.5% | edge +0.074R | POSITIVE
  RR 3.5 : WR 22.6% | edge +0.092R | POSITIVE
  RR 4.0 : WR 20.4% | edge +0.125R | POSITIVE
  RR 4.5 : WR 18.0% | edge +0.133R | POSITIVE
  RR 5.0 : WR 15.0% | edge +0.105R | POSITIVE

── DAY DISTRIBUTION at RR=1.5 (single-account) ──
  trading days: 161 | WIN days: 95 (59%) | LOSE days: 66 (41%)
  ~per 5-day week: 3.0 win / 2.0 lose
  avg net_R: win-day +2.13 | lose-day -2.24 | overall +0.340
  daily net_R: median +1.00 | p25 -1.50 | p75 +2.00 | worst -11.50 | best +6.50

── ROTATION 5-slot (MAE-breach app

In [65]:
from Backtest import ledger_for_rr

R, days = ledger_for_rr(E, 1.5)     # the real array at RR=1.5
print(len(R))                        # how many trades
print(R[:30])                        # first 30 actual R values
print(R.mean())                      # should be ~+0.065, confirming the right array

1978
[ 1.5 -1.  -1.   1.5  1.5 -1.  -1.   1.5 -1.  -1.  -1.  -1.  -1.   1.5
  1.5  1.5 -1.  -1.  -1.  -1.  -1.  -1.  -1.   1.5  1.5 -1.  -1.  -1.
 -1.  -1. ]
0.06488372093023256


In [66]:
import numpy as np
wins     = np.sum(R == 1.5)
stops    = np.sum(R == -1.0)
timeouts = np.sum((R != 1.5) & (R != -1.0))
print(f"wins  (+1.5): {wins:4d}  ({wins/len(R):.1%})")
print(f"stops (-1.0): {stops:4d}  ({stops/len(R):.1%})")
print(f"timeouts    : {timeouts:4d}  ({timeouts/len(R):.1%})")
print(f"timeout values (sample): {R[(R != 1.5) & (R != -1.0)][:20]}")

wins  (+1.5):  830  (42.0%)
stops (-1.0): 1126  (56.9%)
timeouts    :   22  (1.1%)
timeout values (sample): [-0.23  0.8   0.72  0.31  0.66  0.14  0.1  -0.38  0.18  0.63  1.09 -0.03
 -0.22  0.5   0.69 -0.05  1.12  0.69  0.9   1.33]


In [55]:
import pandas as pd, numpy as np

strat = __import__("strategies.dual_kernel_v2", fromlist=["get_strategy"]).get_strategy(**FIXED_DK)
window = bars_ks.loc["2025-01-01":]
STOP, MAXHOLD, FLIP_PTS = 25, 60, 15.0

def flip_sequence(bars, idx, dirs, stop, maxhold, flip_pts):
    """Single-position sequence. flip_pts=inf -> pure HOLD (no flips).
    Finite flip_pts -> close+reverse on strong opposite signals."""
    c=bars["close"].to_numpy(float); n=c.size
    idx=np.asarray(idx); dirs=np.asarray(dirs); order=np.argsort(idx); idx,dirs=idx[order],dirs[order]
    seq=[]; i=0
    while i<len(idx):
        e,k=idx[i],dirs[i]; natural=e+maxhold; forced=None
        for j2 in range(i+1,len(idx)):
            if idx[j2]>natural: break
            if dirs[j2]==-k:
                wm=abs(c[idx[j2]]-c[idx[j2]-2]) if idx[j2]>=2 else 0.0
                if wm>=flip_pts: forced=(j2,idx[j2]); break
        exit_cap = forced[1] if forced else natural
        seq.append((e,k,exit_cap))
        if forced: i=forced[0]
        else:
            while i+1<len(idx) and idx[i+1]<=natural: i+=1
            i+=1
    return seq

def resolve_seq(bars, seq, stop, rr):
    c=bars["close"].to_numpy(float); hi=bars["high"].to_numpy(float); lo=bars["low"].to_numpy(float); n=c.size
    R=[]
    for e,k,cap in seq:
        entry=c[e]; Rv=None
        for j in range(e+1, min(cap,n-1)+1):
            hs=(lo[j]<=entry-stop) if k==1 else (hi[j]>=entry+stop)
            ht=(hi[j]>=entry+rr*stop) if k==1 else (lo[j]<=entry-rr*stop)
            if hs and ht: Rv=-1.0; break
            if hs: Rv=-1.0; break
            if ht: Rv=rr; break
        if Rv is None: j=min(cap,n-1); Rv=float(np.clip((c[j]-entry)/stop*k,-1.0,rr))
        R.append(Rv)
    return np.array(R)

ri, rd = strat.entries(window)

# BOTH through the same resolver — HOLD is flip_pts=inf (never flips), FLIP is finite
print(f"── HOLD vs FLIP (SAME resolver, differ only in flips) | flip_pts={FLIP_PTS} ──")
for rr in [1.0, 1.25, 1.5, 2.0]:
    seq_hold = flip_sequence(window, ri, rd, STOP, MAXHOLD, np.inf)          # no flips
    seq_flip = flip_sequence(window, ri, rd, STOP, MAXHOLD, FLIP_PTS)        # flips
    Rh = resolve_seq(window, seq_hold, STOP, rr)
    Rf = resolve_seq(window, seq_flip, STOP, rr)
    print(f"  RR {rr:<4}: HOLD {len(Rh)}tr edge {Rh.mean():+.3f} | FLIP {len(Rf)}tr edge {Rf.mean():+.3f} | "
          f"flip adds {len(Rf)-len(Rh)}tr | Δedge {Rf.mean()-Rh.mean():+.3f}")

── HOLD vs FLIP (SAME resolver, differ only in flips) | flip_pts=15.0 ──
  RR 1.0 : HOLD 821tr edge +0.096 | FLIP 2683tr edge +0.033 | flip adds 1862tr | Δedge -0.063
  RR 1.25: HOLD 821tr edge +0.085 | FLIP 2683tr edge +0.030 | flip adds 1862tr | Δedge -0.055
  RR 1.5 : HOLD 821tr edge +0.098 | FLIP 2683tr edge +0.024 | flip adds 1862tr | Δedge -0.075
  RR 2.0 : HOLD 821tr edge +0.070 | FLIP 2683tr edge +0.001 | flip adds 1862tr | Δedge -0.069


### 4) Risk Geometry Walkforward 

In [25]:
from Walkforward import walk_forward
strategy_cfg = STRATEGIES[STRATEGY]
bars = load_ohlc(
    DATA_PATH,
    rth_only=strategy_cfg["rth_only"],
    timeframe=strategy_cfg["timeframe"],
    tz="America/New_York",
)

In [13]:
FIXED_DK = dict(hF=4.0, x0F=1, rF=8.0, hS=16.0, x0S=3, rS=8.0, atr_len=14,
                slope_len_f=1, slope_min_f=0.0, slope_len_s=3, slope_min_s=0.0,
                min_body_pts=5.0, use_wick=True,
                scan_at=True, scan_p1=True, scan_p2=True,
                use_ext=False, ext_bars=3, ext_pts=15.0,
                use_er=False, er_len=10, er_min=0.3, use_cq=False, open_min=0.10,
                min_gap_bars=2, sess_start="09:30", sess_end="11:15",
                allow_long=True, allow_short=True, src="close")
def factory_kregime(**swept):
    return __import__("strategies.dual_kernel_v2", fromlist=["get_strategy"]).get_strategy(**{**FIXED_DK, **swept})

PARAM_GRID_KR = {}                       # indicators fixed — nothing swept on the strategy
STOP_GRID  = [20, 25, 30, 35, 40, 45]                        # swept
RISK_PCTS  = [d / FIRM.start_balance * 100 for d in (200, 250, 500, 1000)]   # $200/$250/$500/$1000
RR_GRID    = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]   # swept

bars_ks = load_ohlc(DATA_PATH, rth_only=False, timeframe="2min", tz="America/New_York")

wf_rr = walk_forward(
    FIRM, bars_ks, factory_kregime, PARAM_GRID_KR,
    stop_grid=STOP_GRID, risk_pcts=RISK_PCTS, rr_grid=RR_GRID,
    train_months=6, test_months=3, step_months=3, embargo_days=1,
    max_hold=60, n_paths=1000, max_trades=300,
    min_trades=120, min_trades_per_day=0.25, max_days_to_pass=MAX_DAYS_TO_PASS,
    stage_a_top_n=1, selection="floor", multi_n_accounts=5, multi_max_days=25, seed=0)

print(f"\noptimal-geometry walk-forward (fixed indicators; sweep stop/risk/RR) | pooled OOS: "
      f"{wf_rr.pooled_test_pass_pct:.1f}%  rotation: {wf_rr.pooled_multi_pass_rate:.1f}%")
print("selected (stop, $risk, RR) per fold:",
      [(f.stop_points, round(f.risk_pct/100*FIRM.start_balance), f.rr) for f in wf_rr.folds])
wf_rr.table()

[fold 0] 2021-12-01..2022-03-01  params={}  40pt/3R@0.80%  train_floor=24 train_pass=45  -> TEST single=41% (n=163) multi=44% (8P/10B)
[fold 1] 2022-03-01..2022-06-01  params={}  40pt/3R@0.80%  train_floor=19 train_pass=46  -> TEST single=22% (n=183) multi=25% (4P/12B)
[fold 2] 2022-06-01..2022-09-01  params={}  40pt/2.5R@2.00%  train_floor=17 train_pass=38  -> TEST single=33% (n=165) multi=34% (23P/45B)


KeyboardInterrupt: 

### 5) High-Impact News Reversion Modelling 

In [92]:
import pandas as pd, numpy as np
from pandas_datareader import data as pdr
import requests

# ── FRED release dates for high-impact events ──
# Each FRED "release" has an id; we pull its historical release dates.
# Release times are standardized (ET): 8:30 for CPI/NFP/PPI/PCE/Retail, 10:00 ISM, 14:00 FOMC.
FRED_RELEASES = {
    10:  ("CPI",        "08:30"),   # Consumer Price Index
    50:  ("NFP",        "08:30"),   # Employment Situation (nonfarm payrolls)
    46:  ("PPI",        "08:30"),   # Producer Price Index
    54:  ("PCE",        "08:30"),   # Personal Income & Outlays (PCE)
    53:  ("GDP",        "08:30"),   # GDP
    9:   ("RetailSales","08:30"),   # Advance Retail Sales
}
FRED_KEY = "95d04b388119b66203089daf9dbe295f"   # free at fred.stlouisfed.org/docs/api/api_key.html

def fred_release_dates(release_id, start="2021-01-01", quarterly=False):
    url = (f"https://api.stlouisfed.org/fred/release/dates"
           f"?release_id={release_id}&api_key={FRED_KEY}&file_type=json"
           f"&realtime_start={start}&include_release_dates_with_no_data=false&limit=1000")
    j = requests.get(url).json()
    dates = pd.to_datetime([d["date"] for d in j.get("release_dates", [])]).sort_values()
    s = dates.to_series()
    if quarterly:
        # GDP: 3 estimates per quarter -> keep only the FIRST (advance) per quarter
        return s.groupby([dates.year, dates.quarter]).min().dt.date
    else:
        # monthly: one event per month -> first release of each month (drops revisions)
        return s.groupby([dates.year, dates.month]).min().dt.date

events = []
for rid, (name, t) in FRED_RELEASES.items():
    is_gdp = (name == "GDP")
    for d in fred_release_dates(rid, quarterly=is_gdp):
        ts = pd.Timestamp(f"{d} {t}", tz="America/New_York")
        if ts >= pd.Timestamp("2021-01-01", tz="America/New_York"):
            events.append({"event": name, "time": ts})

# ── FOMC dates (hardcoded — 8/year, decision at 14:00 ET) ──
FOMC = ["2021-01-27","2021-03-17","2021-04-28","2021-06-16","2021-07-28","2021-09-22","2021-11-03","2021-12-15",
        "2022-01-26","2022-03-16","2022-05-04","2022-06-15","2022-07-27","2022-09-21","2022-11-02","2022-12-14",
        "2023-02-01","2023-03-22","2023-05-03","2023-06-14","2023-07-26","2023-09-20","2023-11-01","2023-12-13",
        "2024-01-31","2024-03-20","2024-05-01","2024-06-12","2024-07-31","2024-09-18","2024-11-07","2024-12-18",
        "2025-01-29","2025-03-19","2025-05-07","2025-06-18","2025-07-30","2025-09-17","2025-10-29","2025-12-10",
        "2026-01-28","2026-03-18","2026-04-29","2026-06-17"]
for d in FOMC:
    events.append({"event": "FOMC", "time": pd.Timestamp(f"{d} 14:00", tz="America/New_York")})

ev = pd.DataFrame(events).sort_values("time").reset_index(drop=True)
print(f"{len(ev)} high-impact events 2021-2026")
print(ev.event.value_counts())

398 high-impact events 2021-2026
event
NFP            67
CPI            67
RetailSales    67
PPI            66
PCE            64
FOMC           44
GDP            23
Name: count, dtype: int64


In [98]:
# use 1-min bars for precise event-window measurement
bars_1m = load_ohlc(DATA_PATH, rth_only=False, timeframe="1min", tz="America/New_York")

def study_event(T, spike_win=5, end_win=70):
    idx = bars_1m.index
    after = idx[idx >= T]
    if len(after) == 0: return None
    T = after[0]
    pos = idx.get_loc(T)
    if pos < 1 or pos + end_win >= len(idx): return None
    pre   = bars_1m["close"].iloc[pos-1]
    spike_px = bars_1m["close"].iloc[pos+spike_win]
    end_px   = bars_1m["close"].iloc[pos+end_win]
    spike = spike_px - pre
    if abs(spike) < 1e-9: return None
    rev_frac = (spike_px - end_px) / spike     # >0 reverted toward pre; 1 = full reversion
    return dict(spike_abs=abs(spike), rev_frac=rev_frac, reverted=rev_frac > 0.3)

rows = []
for _, e in ev.iterrows():
    r = study_event(e["time"])
    if r: rows.append({**r, "event": e["event"]})
study = pd.DataFrame(rows)

print(f"\n{len(study)} events with clean windows")
print(f"OVERALL: reverted (>30% retrace) {study.reverted.mean():.0%} | mean rev_frac {study.rev_frac.mean():.2f}")
print("\nby spike size tercile:")
study["spike_bucket"] = pd.qcut(study.spike_abs, 3, labels=["small","med","large"])
print(study.groupby("spike_bucket", observed=True).agg(
    n=("reverted","size"), pct_reverted=("reverted","mean"), mean_rev_frac=("rev_frac","mean")).round(2))
print("\nby event type:")
print(study.groupby("event").agg(
    n=("reverted","size"), pct_reverted=("reverted","mean"), mean_rev_frac=("rev_frac","mean")).round(2))


361 events with clean windows
OVERALL: reverted (>30% retrace) 46% | mean rev_frac -0.01

by spike size tercile:
                n  pct_reverted  mean_rev_frac
spike_bucket                                  
small         121          0.45          -0.28
med           119          0.48          -0.02
large         121          0.44           0.26

by event type:
              n  pct_reverted  mean_rev_frac
event                                       
CPI          61          0.38          -0.27
FOMC         40          0.45           0.91
GDP          21          0.43          -1.49
NFP          61          0.46          -0.78
PCE          57          0.47           0.71
PPI          60          0.50          -0.25
RetailSales  61          0.49           0.47


In [101]:
import pandas as pd, numpy as np
from Backtest import ledger_for_rr, measured_winrate

# strategy with NO session gate — the news window defines when entries are allowed
FIXED_KR_NEWS = dict(FIXED_KR, sess_start=None, sess_end=None)
base = __import__("strategies.kernel_er", fromlist=["get_strategy"]).get_strategy(**FIXED_KR_NEWS)

# unique event times, sorted (dedup same-timestamp releases)
event_times = pd.DatetimeIndex(ev["time"]).drop_duplicates().sort_values()
# map time -> event name(s), handling duplicates by taking first
ev_name = ev.drop_duplicates("time").set_index("time")["event"]

bars_1m = load_ohlc(DATA_PATH, rth_only=False, timeframe="1min", tz="America/New_York")
window = bars_1m.loc["2021-06-28":]

class NewsWindow:
    wants_bars = True
    def __init__(self, inner, events, lo, hi, name):
        self.inner=inner; self.events=events; self.lo=lo; self.hi=hi; self.name=name; self.assigned=[]
    def entries(self, bars):
        idx, dirn = self.inner.entries(bars)
        ts = bars.index[idx]; keep=np.zeros(len(idx),bool); self.assigned=[]
        for k,t in enumerate(ts):
            past=self.events[self.events<=t]
            if len(past)==0: continue
            T=past[-1]; dt=(t-T).total_seconds()/60.0
            if self.lo<=dt<=self.hi:
                keep[k]=True; self.assigned.append((t,T,round(dt)))
        return idx[keep], dirn[keep]

STOP, RR = 25, 1.5

def run(strat, label):
    E = backtest_entries(strat, window, stop_points=STOP, max_hold=30, detail=True, one_position=False)
    n=len(E.day_ids)
    if n==0: print(f"{label}: no trades"); return None, None
    R,_ = ledger_for_rr(E, RR)
    print(f"{label}: {n} tr | WR {measured_winrate(E,RR):.1%} | edge {R.mean():+.3f}R")
    return E, strat

s1 = NewsWindow(base, event_times, 0, 60, "s1")
E1,_ = run(s1, "S1: enter at release .. +60m ")
s2 = NewsWindow(base, event_times, 5, 60, "s2")
E2,_ = run(s2, "S2: enter +5m .. +60m        ")

# ── sample entries per event type per year ──
def sample_table(nw, tag):
    if not nw.assigned:
        print(f"\n{tag}: no entries"); return
    m = pd.DataFrame(nw.assigned, columns=["entry","release","min_after"])
    m["event"] = m["release"].map(ev_name.to_dict())   # dict map avoids the unique-index error
    m["year"]  = m["entry"].dt.year
    print(f"\n─── {tag} sample entries by event & year ───")
    for (evt,yr), g in m.groupby(["event","year"]):
        eg = ", ".join(g["entry"].dt.strftime("%m-%d %H:%M").head(4))
        print(f"  {str(evt):12s} {yr}: {len(g):2d} | e.g. {eg}")

sample_table(s1, "S1")
sample_table(s2, "S2")

C:\\Users\\2115h\\Desktop\\Trading\Propfirm Convex Analysis\Rules\strategies\kernel_er.py:76: RuntimeWarning: invalid value encountered in divide
  er[length:] = np.where(path > 0, net / path, 0.0)


S1: enter at release .. +60m : 1122 tr | WR 32.9% | edge -0.039R


C:\\Users\\2115h\\Desktop\\Trading\Propfirm Convex Analysis\Rules\strategies\kernel_er.py:76: RuntimeWarning: invalid value encountered in divide
  er[length:] = np.where(path > 0, net / path, 0.0)


S2: enter +5m .. +60m        : 899 tr | WR 33.6% | edge -0.036R

─── S1 sample entries by event & year ───
  CPI          2021: 12 | e.g. 07-13 08:30, 07-13 08:40, 07-13 08:59, 07-13 09:30
  CPI          2022: 50 | e.g. 01-12 08:30, 01-12 08:35, 01-12 08:53, 01-12 09:23
  CPI          2023: 33 | e.g. 01-12 08:34, 01-12 08:49, 01-12 09:07, 01-12 09:30
  CPI          2024: 36 | e.g. 01-11 08:30, 01-11 08:45, 01-11 09:15, 02-09 08:31
  CPI          2025: 39 | e.g. 01-15 08:30, 01-15 08:46, 01-15 08:51, 01-15 09:00
  CPI          2026: 24 | e.g. 01-13 08:36, 01-13 09:27, 02-13 08:41, 02-13 08:46
  FOMC         2021: 19 | e.g. 07-28 14:00, 07-28 14:07, 07-28 14:27, 07-28 14:35
  FOMC         2022: 48 | e.g. 01-26 14:10, 01-26 14:25, 01-26 14:31, 01-26 14:38
  FOMC         2023: 42 | e.g. 02-01 14:00, 02-01 14:05, 02-01 14:16, 02-01 14:31
  FOMC         2024: 43 | e.g. 01-31 14:00, 01-31 14:17, 01-31 14:32, 01-31 14:39
  FOMC         2025: 49 | e.g. 01-29 14:00, 01-29 14:11, 01-29 14:17, 01-